# Re:Camp Colab Blender v005 test without Google Drive

Google Drive 권한 승인이 보안 프로그램에 막힐 때 사용하는 대체 Notebook이다. `re-camp`의 CH101 기준 시트와 `re-camp-blender`의 v005 기술 준비 스크립트를 읽고, Colab `/content` 임시 공간에서 Blender를 설치·실행한다. UV·transform·material slot·triangle·LOD0 상태를 JSON으로 확인한 뒤 결과를 ZIP으로 만들어 브라우저로 직접 다운로드한다.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import zipfile
from IPython.display import Image, display

SOURCE_REPO_URL = 'https://github.com/siri2677/re-camp.git'
TOOLS_REPO_URL = 'https://github.com/siri2677/re-camp-blender.git'
BRANCH = 'art/current-roster-gate-a-ch102'
SOURCE_DIR = Path('/content/re-camp')
TOOLS_DIR = Path('/content/re-camp-blender')
OUTPUT_DIR = Path('/content/re-camp-output/CH101_v005')
ZIP_PATH = Path('/content/re-camp-CH101-blockout-v005.zip')
print('No Google Drive access is used.')
print('Source repository:', SOURCE_REPO_URL, BRANCH)
print('Tools repository:', TOOLS_REPO_URL)

In [ ]:
for path in (SOURCE_DIR, TOOLS_DIR):
    if path.exists():
        shutil.rmtree(path)
subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', BRANCH, SOURCE_REPO_URL, str(SOURCE_DIR)
], check=True)
subprocess.run([
    'git', 'clone', '--depth', '1', TOOLS_REPO_URL, str(TOOLS_DIR)
], check=True)
print('Source ready:', SOURCE_DIR)
print('Tools ready:', TOOLS_DIR)

In [ ]:
if shutil.which('blender') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', 'blender'], check=True)
version = subprocess.run(['blender', '--version'], check=True, capture_output=True, text=True)
print((version.stdout or version.stderr).splitlines()[0])

In [ ]:
REFERENCE = SOURCE_DIR / 'art_refs/characters/rin/concept/CH101_Rin_CharacterSheet_APPROVED_v001.png'
BUILD_SCRIPT = TOOLS_DIR / 'scripts/blender/build_blockout.py'
VALIDATE_SCRIPT = TOOLS_DIR / 'scripts/blender/validate_asset.py'
if not REFERENCE.is_file():
    raise FileNotFoundError(REFERENCE)
display(Image(filename=str(REFERENCE), width=420))
print('Source:', REFERENCE)
print('Source commit lock: 418ef96')
print('Blockout revision: v005')

In [ ]:
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
launcher = ['xvfb-run', '-a'] if shutil.which('xvfb-run') else []
run = subprocess.run(launcher + [
    'blender', '--background', '--python', str(BUILD_SCRIPT), '--',
    '--character', 'CH101',
    '--source-asset', str(REFERENCE),
    '--source-commit', '418ef96',
    '--output-dir', str(OUTPUT_DIR),
    '--render', '--export-fbx',
], capture_output=True, text=True)
print('Launcher:', launcher or ['blender'])
print(run.stdout[-5000:])
if run.returncode != 0:
    print(run.stderr[-5000:])
    raise RuntimeError(f'Blender blockout failed: {run.returncode}')

In [ ]:
for view in ('front', 'side', 'back'):
    image_path = OUTPUT_DIR / 'renders' / f'{view}.png'
    if image_path.exists():
        print(view)
        display(Image(filename=str(image_path), width=320))
    else:
        print('Missing render:', image_path)

In [ ]:
blend_path = OUTPUT_DIR / 'CH101_Blockout_REVIEW_v005.blend'
validation_report = OUTPUT_DIR / 'reports' / 'CH101_Blockout_validation_v005.json'
launcher = ['xvfb-run', '-a'] if shutil.which('xvfb-run') else []
run = subprocess.run(launcher + [
    'blender', '--background', '--python', str(VALIDATE_SCRIPT), '--',
    '--blend', str(blend_path),
    '--report', str(validation_report),
], capture_output=True, text=True)
print(run.stdout[-5000:])
if run.returncode != 0:
    print(run.stderr[-5000:])
    raise RuntimeError(f'Blockout validation failed: {run.returncode}')

In [ ]:
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(OUTPUT_DIR))
print('ZIP created:', ZIP_PATH)
print('Download this ZIP before the Colab session ends.')

In [ ]:
from google.colab import files
files.download(str(ZIP_PATH))

## 판정 경계

이 Notebook은 문서용 중립 Blockout·렌더·FBX·JSON 검증을 테스트한다. 결과는 최종 캐릭터, Unity Import, Android 성능, Gate B 승인으로 해석하지 않는다.